<div style="text-align: center;">

# Hazard Data: Request, Transform, and Visualize Point-Level Climate Data

</div>

This notebook shows the end-to-end workflow for requesting point-level hazard data and visualizing the returned intensities on a map.

The workflow is:

1. Load API configuration and credentials from the local environment.
2. Query the API to discover available hazard sources, indicators, scenarios, and years.
3. Generate an example set of latitude/longitude points around London.
4. Build a multi-scenario and multi-year hazard-data request for those points.
5. Submit the request to `/api/get_hazard_data`.
6. Inspect the returned intensity curves.
7. Flatten the curves into a return-period table.
8. Render an interactive map for one scenario and year.

The example requests `CoastalInundation` using the `flood_depth` indicator. It compares historical, `rcp4p5`, and `rcp8p5` conditions for a small synthetic point set near London.

## 1. Imports

The notebook uses `requests` for API calls, `numpy` and `pandas` for point generation and tabular transformations, and `plotly` for interactive map visualization.

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from dotenv import load_dotenv
from IPython.display import HTML

## 2. Load API Configuration

The API base URL and API key are loaded from `../.env`. The API key is sent in the `X-API-Key` header for each API request.

Expected environment variables:

- `ALPHA_KLIMA_API_KEY`: API key used to authenticate requests.
- `ALPHA_KLIMA_API_BASE_URL`: base URL of the Alpha-Klima API, including the host that exposes `/api/get_available_sources` and `/api/get_hazard_data`.


In [2]:
_ = load_dotenv("../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")

## 3. Discover Available Hazard Sources

Before requesting hazard values, query `/api/get_available_sources` to check which hazard sources can be requested with the current API key and use case.

The response is stored as `dict_available_sources`. It contains metadata grouped by hazard family and indicator, including supported scenario-year combinations and the asset types for which each indicator is available.

The helper in the next cell converts that nested response into a compact table so the available hazards, indicators, scenario-year combinations, and supported asset types are easier to inspect.

In [3]:
request = {
"include_all": False,
"use_case_id": "ECB_SCORES",
"selected_hazards_list": [],
}
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

# get available hazard data
available_sources = requests.post(
    url=f"{API_BASE}/api/get_available_sources", json=request, headers=headers
)

dict_available_sources = available_sources.json()

In [4]:
def available_sources_to_table(available_sources: dict[str, dict]) -> pd.DataFrame:
    rows = []

    hazards: list[str] = list(available_sources["hazards"].keys())

    for h in hazards:
        h_dict: dict[str, dict] = available_sources["hazards"][h]
        indicators: list[str] = list(h_dict.keys())

        for ind in indicators:
            available_data: dict[str, list | str] = h_dict[ind]

            available_data_scenarios: list[dict] = available_data["scenarios"]
            scenarios_n_years: list[dict] = [{"scenario": d["id"], "years": d["years"]} for d in available_data_scenarios]

            asset_types: list[str] = available_data['available_for_assets_type']

            rows.append(
                {
                    "hazard": h,
                    "indicator": ind,
                    "scenario-year": scenarios_n_years,
                    "asset_types": asset_types
                }
            )

    return pd.DataFrame(rows)

In [5]:
available_sources_to_table(dict_available_sources)

,hazard,indicator,scenario-year,asset_types
0,Wind,max_speed,"[{'scenario': 'historical', 'years': [2010]}, ...","[UtilityAsset, RealEstateAsset, IndustrialActi..."
1,Wind,wind_speed/3s,"[{'scenario': 'historical', 'years': [1999]}]","[RealEstateAsset, TelecommunicationAsset, Powe..."
2,Landslide,landslide_susceptability,"[{'scenario': 'historical', 'years': [2018]}]","[UtilityAsset, RealEstateAsset, IndustrialActi..."
3,Subsidence,subsidence_susceptability,"[{'scenario': 'historical', 'years': [1980]}]","[RealEstateAsset, TelecommunicationAsset, Ther..."
4,RiverineInundation,flood_depth,"[{'scenario': 'historical', 'years': [1985]}, ...","[RealEstateAsset, InfrastructureAsset, Telecom..."
5,CoastalInundation,flood_depth,"[{'scenario': 'historical', 'years': [1971]}, ...","[RealEstateAsset, InfrastructureAsset, Telecom..."
6,ChronicHeat,mean_degree_days/above/index,"[{'scenario': 'historical', 'years': [2005]}, ...","[RealEstateAsset, TelecommunicationAsset]"
7,ChronicHeat,mean_degree_days/above/32c,"[{'scenario': 'historical', 'years': [2005]}, ...",[IndustrialActivity]
8,ChronicHeat,mean_work_loss/high,"[{'scenario': 'historical', 'years': [2005]}, ...",[IndustrialActivity]
9,Fire,fire_probability,"[{'scenario': 'historical', 'years': [2010]}, ...","[RealEstateAsset, InfrastructureAsset, Telecom..."


## 4. Define Locations of Interest

For demonstration purposes, this notebook creates a reproducible sample of random points around London. These coordinates are the locations where hazard intensity curves will be requested.

In a real workflow, replace this synthetic point set with coordinates from an asset register, exposure dataset, CSV file, database, or geospatial pipeline. The API request expects latitude and longitude arrays with matching order and length.

In [6]:
# Center point (London)
center_lat = 51.5155
center_lon = 0.0495

# ~5–6 km N/S and ~5–6 km E/W box (0.05 deg ≈ 5.5 km in lat; lon similar near London)
dlat = 0.05
dlon = 0.08  # a tad wider east-west to cover more of the Thames corridor

rng = np.random.default_rng(7)
n = 50

lats = rng.uniform(center_lat - dlat, center_lat + dlat, n)
lons = rng.uniform(center_lon - dlon, center_lon + dlon, n)


points = pd.DataFrame({"lat": lats, "lon": lons})
print(points.head())

         lat       lon
0  51.528010  0.027302
1  51.555221  0.065209
2  51.543069 -0.021020
3  51.488021  0.031521
4  51.495517  0.021186


## 5. Build the Hazard Data Request

The hazard-data endpoint accepts a request dictionary with an `items` list. Each item asks for one hazard, indicator, scenario, and year combination evaluated at the same set of points.

Each request item includes:

- `request_item_id`: a label used to identify the result in the response.
- `event_type`: the hazard family, here `CoastalInundation`.
- `indicator_id`: the requested measure, here `flood_depth`.
- `scenario`: the climate scenario, such as `historical`, `rcp4p5`, or `rcp8p5`.
- `year`: the historical baseline or future horizon.
- `latitudes` and `longitudes`: the point coordinates to evaluate.

In [7]:
# Create hazard request
request_dict = {
    "items": [
        {
            "request_item_id": "coastal_rcp4p5_2035",
            "event_type": "CoastalInundation",
            "longitudes": points["lon"].to_list(),
            "latitudes": points["lat"].to_list(),
            "year": 2035,
            "scenario": "rcp4p5",
            "indicator_id": "flood_depth",
        },
        {
            "request_item_id": "coastal_rcp4p5_2085",
            "event_type": "CoastalInundation",
            "longitudes": points["lon"].to_list(),
            "latitudes": points["lat"].to_list(),
            "year": 2085,
            "scenario": "rcp4p5",
            "indicator_id": "flood_depth",
        },
        {
            "request_item_id": "coastal_rcp8p5_2035",
            "event_type": "CoastalInundation",
            "longitudes": points["lon"].to_list(),
            "latitudes": points["lat"].to_list(),
            "year": 2035,
            "scenario": "rcp8p5",
            "indicator_id": "flood_depth",
        },
        {
            "request_item_id": "coastal_rcp8p5_2085",
            "event_type": "CoastalInundation",
            "longitudes": points["lon"].to_list(),
            "latitudes": points["lat"].to_list(),
            "year": 2085,
            "scenario": "rcp8p5",
            "indicator_id": "flood_depth",
        },
        {
            "request_item_id": "coastal_historical",
            "event_type": "CoastalInundation",
            "longitudes": points["lon"].to_list(),
            "latitudes": points["lat"].to_list(),
            "year": 1971,
            "scenario": "historical",
            "indicator_id": "flood_depth",
        }
    ],
}

## 6. Execute the Hazard Data Request

The request is submitted to `/api/get_hazard_data`. The JSON response is stored as `dict_hazard_data`.

The response contains an `items` list that mirrors the request. For each item, the main payload is usually `intensity_curve_set`:

- There is one curve per requested point.
- `index_values` contains return periods, for example 10, 30, 100, 300, and 1000 years.
- `intensities` contains hazard intensity values aligned with those return periods. In this example, the values represent coastal flood depth in meters.

The summary printout below is a sanity check: it confirms how many point curves were returned for each request item and the minimum-to-maximum intensity range across those curves.

In [8]:
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

# get available hazard data
hazard_data = requests.post(
    url=f"{API_BASE}/api/get_hazard_data", json=request_dict, headers=headers
)

dict_hazard_data = hazard_data.json()

In [9]:
for item in dict_hazard_data["items"]:
    rid = item["request_item_id"]
    scenario = item["scenario"]
    year = item["year"]

    curves = item.get("intensity_curve_set", [])
    n_points = len(curves)

    # Flatten all intensities from all curves
    all_vals = [v for c in curves for v in c.get("intensities", []) if v is not None]

    print(f"{rid} — {scenario} {year}: {n_points} locations", end="")

    if all_vals:
        print(f" | intensity range = {min(all_vals):.2f} → {max(all_vals):.2f} [m]")
    else:
        print(" | no intensity data")

coastal_rcp4p5_2035 — rcp4p5 2035: 50 locations | intensity range = 0.00 → 5.19 [m]
coastal_rcp4p5_2085 — rcp4p5 2085: 50 locations | intensity range = 0.00 → 5.15 [m]
coastal_rcp8p5_2035 — rcp8p5 2035: 50 locations | intensity range = 0.00 → 4.79 [m]
coastal_rcp8p5_2085 — rcp8p5 2085: 50 locations | intensity range = 0.00 → 5.19 [m]
coastal_historical — historical 1971: 50 locations | intensity range = 0.00 → 5.03 [m]


## 7. Convert Intensity Curves to a Return-Period Table

For analysis and visualization, it is easier to work with a flat table than with nested response curves. The helper below converts each curve into one row per point, scenario, and year.

The resulting table contains:

- `lat` and `lon`: point coordinates.
- `scenario` and `year`: the request dimensions.
- `rp10`, `rp30`, `rp100`, `rp300`, and `rp1000`: flood-depth values for selected return periods.

If a selected return period is not present in a curve, the helper fills that value with `NaN`.

In [10]:
#build rp table
def build_rp_table(
    data: dict, points: pd.DataFrame, rp_list=(10.0, 30.0, 100.0, 300.0, 1000.0)
) -> pd.DataFrame:
    """Flatten API response into rows: lat, lon, scenario, year, rp10, rp30, rp100, rp300, rp1000"""
    rows = []
    for item in data.get("items", []):
        curves = item["intensity_curve_set"]
        scen, year = item.get("scenario"), item.get("year")
        # one curve per input point
        mat = []
        for c in curves:
            rp_map = {
                rp: iv
                for rp, iv in zip(c.get("index_values", []), c.get("intensities", []))
            }
            mat.append([rp_map.get(rp, np.nan) for rp in rp_list])
        arr = np.asarray(mat, float)
        df = points.reset_index(drop=True).copy()
        df["scenario"], df["year"] = scen, year
        for j, rp in enumerate(rp_list):
            df[f"rp{int(rp)}"] = arr[:, j]
        rows.append(df)
    return pd.concat(rows, ignore_index=True)

In [11]:
df_rp = build_rp_table(dict_hazard_data, points)

df_rp

,lat,lon,scenario,year,rp10,rp30,rp100,rp300,rp1000
0,51.528010,0.027302,rcp4p5,2035,NaN,NaN,0.000,NaN,NaN
1,51.555221,0.065209,rcp4p5,2035,NaN,NaN,0.000,NaN,NaN
2,51.543069,-0.021020,rcp4p5,2035,NaN,NaN,0.000,NaN,NaN
3,51.488021,0.031521,rcp4p5,2035,NaN,NaN,NaN,NaN,0.5306
4,51.495517,0.021186,rcp4p5,2035,3.666,3.916,4.185,4.428,4.6940
...,...,...,...,...,...,...,...,...,...
245,51.539677,0.089712,historical,1971,NaN,NaN,0.000,NaN,NaN
246,51.474650,-0.026469,historical,1971,NaN,NaN,0.000,NaN,NaN
247,51.519614,0.029050,historical,1971,NaN,NaN,0.000,NaN,NaN
248,51.516277,-0.025644,historical,1971,NaN,NaN,0.000,NaN,NaN


## 8. Visualize Results on an Interactive Map

The mapping helper creates a point map for one `(scenario, year)` slice of `df_rp`. It uses marker size and color to show hazard intensity at different return periods.

Encoding choices in this notebook:

- Marker size is driven by `rp10`, with fallbacks to `rp30` and then `rp100`.
- Marker color is driven by `rp100`, with fallbacks to `rp300` and then `rp30`.

These defaults highlight both frequent flooding (`rp10`) and more severe flooding (`rp100`). You can adjust the function if another return period is more relevant for your analysis.

In [12]:
#build interactive map
def coastal_abs_map(
    df_rp: pd.DataFrame,
    scenario: str,
    year: int,
    map_style="open-street-map",
    zoom=11.5,
    height=520,
) -> go.Figure:
    """Absolute view: size=RP10 (fallback RP30→RP100), color=RP100 (fallback RP300→RP30)."""
    PALETTE = {
    "pri": "#0092A0",
    "dark": "#004D73",
    "acc": "#E15E0B",
    "blue2": "#3A84A0",
    "bg": "white",
    }
    COLOR_SCALE = [[0, "#3A84A0"], [0.5, "#0092A0"], [1, "#E15E0B"]]
    sub = df_rp[(df_rp.scenario == scenario) & (df_rp.year == year)].copy()

    # ---- Fallbacks to avoid NaN in size/color ----
    # size prefers frequent hazard: RP10 → RP30 → RP100
    size_val = sub["rp10"].where(
        sub["rp10"].notna(), sub["rp30"].where(sub["rp30"].notna(), sub["rp100"])
    )
    # color prefers severe hazard: RP100 → RP300 → RP30
    color_val = sub["rp100"].where(
        sub["rp100"].notna(), sub["rp300"].where(sub["rp300"].notna(), sub["rp30"])
    )

    # Replace remaining NaNs with 0, and clip to non-negative
    sub["_size"] = size_val.fillna(0).clip(lower=0)
    sub["_color"] = color_val.fillna(0).clip(lower=0)

    fig = px.scatter_map(
        sub,
        lat="lat",
        lon="lon",
        color="_color",
        size="_size",
        color_continuous_scale=COLOR_SCALE,
        size_max=18,
        zoom=zoom,
        height=height,
        hover_data={
            "rp10": ":.2f",
            "rp30": ":.2f",
            "rp100": ":.2f",
            "rp300": ":.2f",
            "rp1000": ":.2f",
        },
    )
    fig.update_layout(
        map_style=map_style,
        map_center={"lat": float(sub.lat.mean()), "lon": float(sub.lon.mean())},
        title="Coastal inundation — Absolute view<br><span style='font-size:12px'>Size: RP10 (fallbacks) | Color: RP100 (fallbacks)</span>",
        coloraxis_colorbar=dict(title="RP100"),
        margin=dict(l=10, r=10, t=70, b=10),
        paper_bgcolor=PALETTE["bg"],
    )

    fig.update_layout(
        font=dict(
            family="Source Sans Pro, Open Sans, Arial",
            size=12,
            color="#004D73",  # your dark palette color for text
        ),
        title_font=dict(
            size=14, family="Source Sans Pro, Open Sans, Arial", color="#0092A0"
        ),
    )
    return fig

In [13]:
def show_plotly(fig):
    html = pio.to_html(
        fig, include_plotlyjs="cdn", full_html=False
    )  # bundles the figure + loads Plotly JS
    display(HTML(html))

### Render the Example Map

This final cell renders the map for `rcp4p5` in 2035 using the `satellite` basemap. Change `scenario`, `year`, or `map_style` to compare other request items.

In [14]:
fig_2 = coastal_abs_map(df_rp, scenario="rcp4p5", year=2035, map_style="satellite")

show_plotly(fig_2)